# 09d - Ensemble: check for optimal blend of tuned base learners

Blends the tuned base learners (`09c`) three ways: equal-weight, tuned-weight grid search, and a logistic meta-learner (stacking). 
Weights are fit on `val` (2024), evaluated once on `test` (2025) - done on top of already-tuned hyperparameters.

## Setup

In [ ]:
import time as _time

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from sklearn.ensemble import (
    HistGradientBoostingRegressor, RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
)

import xgboost as xgb

from utils import (
    load_model_split, prep_gbm_matrices, GBM_FEATURES, GBM_CAT_FEATURES,
)

BASEPATH = "../data"

# temporary progress instrumentation for long-running cells below --
# tail /tmp/09d_progress.log from a separate shell to check in on a
# background run
PROGRESS_LOG = "/tmp/09d_progress.log"

def log_progress(msg):
    ts = _time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(PROGRESS_LOG, "a") as f:
        f.write(line + "\n")

log_progress("09d_ensemble.ipynb started")

train_df, test_df = load_model_split(BASEPATH)

# kept only for the complete-case mask, matching every other notebook
logit_predictor_cols = [
    "line", "wx_primary_asof_collapsed", "temp_c_asof",
    "wind_speed_ms_asof", "precip_mm_asof", "any_gust_24h",
    "any_ice_24h", "has_cloud_layer", "am_peak_overlap_bin",
    "pm_peak_x_sports", "day_of_week", "is_holiday",
    "sched_duration_sec", "is_inbound", "lag_line_lateness_min",
    "lag_network_lateness_2hr_mean", "lag_train_lateness_5run_mean",
]

VAL_YEAR = 2024
train_sub = train_df[train_df["year"] < VAL_YEAR].copy()
val_df = train_df[train_df["year"] == VAL_YEAR].copy()

val_mask = val_df[logit_predictor_cols].notna().all(axis = 1)
test_mask = test_df[logit_predictor_cols].notna().all(axis = 1)
val_cc, test_cc = val_df[val_mask].copy(), test_df[test_mask].copy()
y_val_cc, y_test_cc = val_cc["is_otp"], test_cc["is_otp"]

print(
    f"train_sub: {train_sub.shape}, val: {val_df.shape}, "
    f"test: {test_df.shape}"
)

## 0. Rebuild the stacked lateness feature (train_sub / val / test)

Same as `09b` and `09c`, extended to all three splits since this notebook needs evaluation. `TUNED_REG_PARAMS` are fixed from `09b`'s `RandomizedSearchCV` vs. researching them here

In [ ]:
TUNED_REG_PARAMS = dict(
    min_samples_leaf = 10, max_leaf_nodes = 127, max_iter = 500,
    max_depth = 7, learning_rate = 0.03, l2_regularization = 1,
)

X_train_sub_gbm, X_val_gbm = prep_gbm_matrices(train_sub, val_df)
_, X_test_gbm_base = prep_gbm_matrices(train_sub, test_df)
y_train_sub_log = np.log1p(train_sub["lateness"])

# out-of-fold predictions on train_sub so each row's stacked feature
# never sees its own lateness during fitting
_t0 = _time.time()
log_progress(
    "Rebuilding out-of-fold stacked lateness feature on "
    "train_sub (5-fold KFold)..."
)
kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
oof_pred_lateness = np.zeros(len(train_sub))
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_sub_gbm), start = 1):
    _tfold = _time.time()
    fold_reg = HistGradientBoostingRegressor(
        categorical_features = "from_dtype", random_state = 42,
        **TUNED_REG_PARAMS
    )
    fold_reg.fit(X_train_sub_gbm.iloc[tr_idx], y_train_sub_log.iloc[tr_idx])
    oof_pred_lateness[val_idx] = np.expm1(
        fold_reg.predict(X_train_sub_gbm.iloc[val_idx])
    )
    log_progress(f"  fold {fold}/5 done ({_time.time() - _tfold:.1f}s)")

train_sub["pred_lateness_gbm"] = oof_pred_lateness

# val/test were never used to fit this regressor, so a plain fit on
# all of train_sub (no OOF needed) is safe for their predictions
reg_full = HistGradientBoostingRegressor(
    categorical_features = "from_dtype", random_state = 42,
    **TUNED_REG_PARAMS
)
reg_full.fit(X_train_sub_gbm, y_train_sub_log)
val_df["pred_lateness_gbm"] = np.expm1(reg_full.predict(X_val_gbm))
test_df["pred_lateness_gbm"] = np.expm1(reg_full.predict(X_test_gbm_base))

STACKED_FEATURES = GBM_FEATURES + ["pred_lateness_gbm"]
val_cc = val_df[val_mask].copy()
test_cc = test_df[test_mask].copy()
log_progress(
    f"Stacked feature rebuilt on train_sub/val/test "
    f"({_time.time() - _t0:.1f}s total)"
)

## 1. Refit logit, tuned XGBoost, tuned RF on train_sub

`TUNED_XGB_PARAMS`/`TUNED_RF_PARAMS` are `09c`'s actual best-found configs (not placeholders).

In [ ]:
# compact predictor set for interpretability, not the full
# GBM_FEATURES list
line_fe_formula = (
    "is_otp ~ "
    "C(line) + "
    "C(wx_primary_asof_collapsed, Treatment(reference='A_CLR')) + "
    "temp_c_asof + wind_speed_ms_asof + precip_mm_asof + "
    "any_gust_24h + any_ice_24h + has_cloud_layer + "
    "C(am_peak_overlap_bin) + "
    "C(pm_peak_x_sports, Treatment(reference='none_x_none')) + "
    "C(day_of_week) + is_holiday + sched_duration_sec + is_inbound + "
    "lag_line_lateness_min + lag_network_lateness_2hr_mean + "
    "lag_train_lateness_5run_mean"
)
_t0 = _time.time()
log_progress("Fitting logit (train_sub)...")
logit_sub = smf.logit(line_fe_formula, data = train_sub).fit(disp = False)
p_logit_val = logit_sub.predict(val_cc)
p_logit_test = logit_sub.predict(test_cc)
log_progress(f"logit fit complete ({_time.time() - _t0:.1f}s)")

In [ ]:
# 09c's current best-search configs (updated 2026-08-07 -- 09c was
# re-run and found different best params than this cell previously had)
TUNED_XGB_PARAMS = dict(
    subsample = 1.0, reg_lambda = 5, n_estimators = 300,
    min_child_weight = 1, max_depth = 7, learning_rate = 0.05,
    colsample_bytree = 1.0,
)
TUNED_RF_PARAMS = dict(
    n_estimators = 200, min_samples_leaf = 20, max_features = 0.5,
    max_depth = 10,
)

X_train_sub_stacked, X_val_stacked = prep_gbm_matrices(
    train_sub, val_df,
    features = STACKED_FEATURES, cat_features = GBM_CAT_FEATURES,
)
_, X_test_stacked = prep_gbm_matrices(
    train_sub, test_df,
    features = STACKED_FEATURES, cat_features = GBM_CAT_FEATURES,
)

# XGBoost splits on categoricals natively -- predict on the full val/
# test matrices, then restrict to the complete-case rows afterward so
# the comparison against logit uses the same rows
_t0 = _time.time()
log_progress("Fitting tuned XGBoost (train_sub)...")
gbm_sub = xgb.XGBClassifier(
    random_state = 42, enable_categorical = True,
    tree_method = "hist", **TUNED_XGB_PARAMS
)
gbm_sub.fit(X_train_sub_stacked, train_sub["is_otp"])
p_gbm_val = gbm_sub.predict_proba(X_val_stacked)[:, 1][val_mask.values]
p_gbm_test = gbm_sub.predict_proba(X_test_stacked)[:, 1][test_mask.values]
log_progress(f"tuned XGBoost fit complete ({_time.time() - _t0:.1f}s)")

# random forest needs categoricals one-hot encoded; align val/test
# columns to train_sub's in case a category is missing from either
X_train_sub_rf = pd.get_dummies(
    train_sub[STACKED_FEATURES], columns = GBM_CAT_FEATURES
)
X_val_rf = pd.get_dummies(
    val_df[STACKED_FEATURES], columns = GBM_CAT_FEATURES
).reindex(columns = X_train_sub_rf.columns, fill_value = 0)
X_test_rf = pd.get_dummies(
    test_df[STACKED_FEATURES], columns = GBM_CAT_FEATURES
).reindex(columns = X_train_sub_rf.columns, fill_value = 0)

_t0 = _time.time()
log_progress(
    "Fitting tuned RF (train_sub) -- likely the slowest cell here..."
)
rf_sub = RandomForestClassifier(
    n_jobs = -1, random_state = 42, **TUNED_RF_PARAMS
)
rf_sub.fit(X_train_sub_rf, train_sub["is_otp"])
p_rf_val = rf_sub.predict_proba(X_val_rf)[:, 1][val_mask.values]
p_rf_test = rf_sub.predict_proba(X_test_rf)[:, 1][test_mask.values]
log_progress(f"tuned RF fit complete ({_time.time() - _t0:.1f}s)")

## 2. Blend three ways

(1) equal-weight average, (2) linear blend with weights grid-searched to maximize validation ROC-AUC, (3) logistic-regression meta-learner (stacking) fit on `val` predictions. All scored honestly on `val` only, then evaluated once on `test`.

In [ ]:
# grid-search blend weights on val only -- w_rf is whatever's left
# after w_logit/w_gbm, clipped to a valid [0, 1] share, so the search
# only needs to loop over two of the three weights
_t0 = _time.time()
log_progress("Grid-searching blend weights on val...")
best_weights, best_val_auc = None, -1
grid = np.arange(0, 1.01, 0.05)
for w_logit in grid:
    for w_gbm in grid:
        w_rf = 1 - w_logit - w_gbm
        if w_rf < -1e-9 or w_rf > 1 + 1e-9:
            continue
        w_rf = max(w_rf, 0)
        blend = w_logit * p_logit_val + w_gbm * p_gbm_val + w_rf * p_rf_val
        auc = roc_auc_score(y_val_cc, blend)
        if auc > best_val_auc:
            best_val_auc, best_weights = auc, (w_logit, w_gbm, w_rf)

equal_weight_val_auc = roc_auc_score(
    y_val_cc, (p_logit_val + p_gbm_val + p_rf_val) / 3
)
log_progress(f"Weight grid search complete ({_time.time() - _t0:.1f}s)")
print(
    f"tuned weights (logit, XGBoost, RF): "
    f"{tuple(round(w, 3) for w in best_weights)}"
)
print(
    f"validation ROC-AUC -- tuned: {best_val_auc:.4f}, "
    f"equal-weight: {equal_weight_val_auc:.4f}"
)

In [ ]:
# stacking: a logistic regression learns its own weights on the three
# base learners' val-set predictions, instead of a hand-searched blend
meta_X_val = np.column_stack([p_logit_val, p_gbm_val, p_rf_val])
meta_learner = LogisticRegression()
meta_learner.fit(meta_X_val, y_val_cc)
print(
    "meta-learner coefficients (logit, XGBoost, RF):",
    np.round(meta_learner.coef_[0], 3),
    "intercept:", round(meta_learner.intercept_[0], 3),
)

## 3. Evaluate all three blends, plus the individual base learners, on test

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, balanced_accuracy_score,
)

def evaluate(name, y_true, p_otp, threshold = 0.5):
    y_late = 1 - y_true
    p_late = 1 - p_otp
    pred = (p_otp >= threshold).astype(int)
    pred_late = 1 - pred
    return {
        "model": name,
        "accuracy": (pred == y_true).mean(),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "roc_auc": roc_auc_score(y_true, p_otp),
        "pr_auc_late": average_precision_score(y_late, p_late),
        "precision_late": precision_score(
            y_late, pred_late, zero_division = 0
        ),
        "recall_late": recall_score(y_late, pred_late, zero_division = 0),
        "f1_late": f1_score(y_late, pred_late, zero_division = 0),
        "brier": brier_score_loss(y_true, p_otp),
    }

# apply each blend's weights/meta-learner to the (held-out) test set
w_logit, w_gbm, w_rf = best_weights
p_ensemble_equal = (p_logit_test + p_gbm_test + p_rf_test) / 3
p_ensemble_tuned = (
    w_logit * p_logit_test + w_gbm * p_gbm_test + w_rf * p_rf_test
)
p_ensemble_meta = meta_learner.predict_proba(
    np.column_stack([p_logit_test, p_gbm_test, p_rf_test])
)[:, 1]

final_results = pd.DataFrame([
    evaluate("logit: line FE (train_sub)", y_test_cc, p_logit_test),
    evaluate("tuned XGBoost (train_sub)", y_test_cc, p_gbm_test),
    evaluate("tuned random forest (train_sub)", y_test_cc, p_rf_test),
    evaluate("ensemble: equal-weight blend", y_test_cc, p_ensemble_equal),
    evaluate("ensemble: tuned-weight blend", y_test_cc, p_ensemble_tuned),
    evaluate(
        "ensemble: logistic meta-learner (stacking)",
        y_test_cc, p_ensemble_meta,
    ),
]).set_index("model")
print(final_results.to_string())
log_progress("09d_ensemble.ipynb complete")

## Decision: XGBoost alone, not the ensemble

The tuned-weight blend (logit=0.0, XGBoost=0.95, RF=0.05) is tied with tuned XGBoost alone on every metric above (0.8321 vs. 0.8322 accuracy, 0.8045 vs. 0.8045 ROC-AUC, 0.610 vs. 0.610 PR-AUC-late, 0.1240 vs. 0.1240 Brier).
5% RF weight is not distinguishable from noise, not evidence of actual helpful signal, and adds unnecessary complexity to the model (both for describing and running later 10_ analyses).

The final model is tuned XGBoost alone, not an ensemble; remaining predictive scripts use it.
Summary: 83.22% accuracy, 0.8045 ROC-AUC (test, tuned XGBoost alone)

## 4. Save artifacts for reuse

Saves expensive outputs so downstream scripts can load vs. refitting: stacked lateness feature (`train_sub`/`val`/`test`), every fitted model, and prediction arrays for every model in the comparison table (`val` and `test`)

In [ ]:
import joblib

_t0 = _time.time()
log_progress("Saving cached artifacts...")

# stacked lateness feature, keyed by original row index
# # future notebooks can left join onto train_sub, val_df, test_df
# don't need to rebuild w/ 5fold OOF
stacked_feature_cache = pd.concat([
    train_sub[["pred_lateness_gbm"]].assign(split = "train_sub"),
    val_df[["pred_lateness_gbm"]].assign(split = "val"),
    test_df[["pred_lateness_gbm"]].assign(split = "test"),
])
stacked_feature_cache.to_parquet(
    f"{BASEPATH}/9_pred_lateness_gbm_cached.parquet"
)

# joblib.load() fitted models
joblib.dump(reg_full, f"{BASEPATH}/9_reg_gbm_model.joblib")
joblib.dump(logit_sub, f"{BASEPATH}/9d_logit_model.joblib")
joblib.dump(gbm_sub, f"{BASEPATH}/9d_final_xgb_model.joblib")
joblib.dump(rf_sub, f"{BASEPATH}/9d_rf_model.joblib")

# prediction arrays for every model in the comparison table
predictions_val = pd.DataFrame({
    "is_otp": y_val_cc.values,
    "p_logit": p_logit_val,
    "p_xgb": p_gbm_val,
    "p_rf": p_rf_val,
}, index = val_cc.index)
predictions_test = pd.DataFrame({
    "is_otp": y_test_cc.values,
    "p_logit": p_logit_test,
    "p_xgb": p_gbm_test,
    "p_rf": p_rf_test,
    "p_ensemble_equal": p_ensemble_equal,
    "p_ensemble_tuned": p_ensemble_tuned,
    "p_ensemble_meta": p_ensemble_meta,
}, index = test_cc.index)
predictions_val.to_parquet(f"{BASEPATH}/9d_predictions_val.parquet")
predictions_test.to_parquet(f"{BASEPATH}/9d_predictions_test.parquet")

# ensemble config w/ weights, meta-learner, tuned hyperp dicts
joblib.dump(
    {
        "weights_logit_xgb_rf": best_weights,
        "meta_learner": meta_learner,
        "TUNED_XGB_PARAMS": TUNED_XGB_PARAMS,
        "TUNED_RF_PARAMS": TUNED_RF_PARAMS,
        "TUNED_REG_PARAMS": TUNED_REG_PARAMS,
    },
    f"{BASEPATH}/9d_ensemble_config.joblib",
)
log_progress(f"Artifact save complete ({_time.time() - _t0:.1f}s)")

import os
saved_files = [
    f for f in os.listdir(BASEPATH) if f.startswith(("9_", "9d_"))
]
print("Cached artifacts now on disk:")
for f in sorted(saved_files):
    size_mb = os.path.getsize(f"{BASEPATH}/{f}") / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

[10:20:49] Saving cached artifacts...


[10:20:51] Artifact save complete (1.7s)
Cached artifacts now on disk:
  9_baseline_predictions_test.parquet (2.8 MB)
  9_pred_lateness_gbm_cached.parquet (18.9 MB)
  9_reg_gbm_model.joblib (7.2 MB)
  9d_ensemble_config.joblib (0.0 MB)
  9d_final_xgb_model.joblib (3.8 MB)
  9d_logit_model.joblib (2977.6 MB)
  9d_predictions_test.parquet (7.9 MB)
  9d_predictions_val.parquet (5.7 MB)
  9d_rf_model.joblib (215.8 MB)
